### DSP

In [ ]:
!mkdir -p eval/out

!yosys -q -p "read_verilog eval/systolic/systolic_matmul_4x4_w8.v; proc; memory_map; opt; write_json eval/out/systolic_matmul_4x4_w8.json; synth_xilinx -family xcup; tee -o eval/out/systolic_matmul_4x4_w8.stat stat"

In [1]:
dsp_rules = {
    "dsp_generic": {
        "requirements": {
            "dsp48e2": 1
        },
        "hidden_inputs": ["clk"],
        "inputs": ["inputs"],
        "outputs": ["outputs"]
    }
}

In [2]:
import emap
import json

SCHEMA_PATH = "emap/schema.sql"
TEST_NAME = "systolic_matmul_4x4_w8"

def simple_cost_model(type_: str, *ports) -> float:
    if type_ == "$dff":
        return len(ports[0]) * 1.0
    elif type_ in {"$muls", "$mulu"}:
        return len(ports[0]) * len(ports[1]) * 1.0
    elif type_ in {"$adds", "$addu", "$subs", "$subu"}:
        return min(len(ports[0]) + len(ports[1]), len(ports[2])) * 1.0
    return len(ports[0]) * 1.0  # other types

netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])
netlist.rebuild()

cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds", "$muls"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$muls"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$muls"])

    cnt = 0
    cnt += emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)

    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.techmap_dsp(netlist)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 16}, OutputFlag=False)
with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

Found 88 cells
Processing cell 0/88: $add$eval/systolic/systolic_matmul_4x4_w8.v:65$460
Database built with 1298 wires and global clock 2
Applied 117 rewrites
Applied 32 rewrites
Applied 6 rewrites
No rewrites applied, stopping
C++ backend emapcc found
Removed 121 dominated cells, 476 remain
C++ backend emapcc found
Grouped 2100 wires into 168 groups
Set parameter Username
Set parameter LicenseID to value 2690590
Academic license - for non-commercial use only - expires 2026-07-24
ILP model solved with objective value: 656.0


In [ ]:
!yosys -q -p "read_json eval/out/systolic_matmul_4x4_w8_extracted.json; read_verilog eval/blackboxes/dsp_defs.v; synth_xilinx -family xcup; tee -o eval/out/systolic_matmul_4x4_w8_extracted.stat stat"

In [ ]:
!yosys -q -p "read_verilog eval/systolic/systolic_matmul_4x4_w16.v; proc; memory_map; opt; write_json eval/out/systolic_matmul_4x4_w16.json; synth_xilinx -family xcup; tee -o eval/out/systolic_matmul_4x4_w16.stat stat"

In [3]:
import emap
import json

SCHEMA_PATH = "emap/schema.sql"
TEST_NAME = "systolic_matmul_4x4_w16"

def simple_cost_model(type_: str, *ports) -> float:
    if type_ == "$dff":
        return len(ports[0]) * 1.0
    elif type_ in {"$muls", "$mulu"}:
        return len(ports[0]) * len(ports[1]) * 1.0
    elif type_ in {"$adds", "$addu", "$subs", "$subu"}:
        return min(len(ports[0]) + len(ports[1]), len(ports[2])) * 1.0
    return len(ports[0]) * 1.0  # other types

netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])
netlist.rebuild()

cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds", "$muls"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$muls"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$muls"])

    cnt = 0
    cnt += emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)

    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.techmap_dsp(netlist)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 16}, OutputFlag=False)
with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

Found 88 cells
Processing cell 0/88: $add$eval/systolic/systolic_matmul_4x4_w16.v:65$460
Database built with 2578 wires and global clock 2
Applied 117 rewrites
Applied 32 rewrites
Applied 6 rewrites
No rewrites applied, stopping
C++ backend emapcc found
Removed 121 dominated cells, 476 remain
C++ backend emapcc found
Grouped 4180 wires into 168 groups
ILP model solved with objective value: 1296.0


In [ ]:
!yosys -q -p "read_json eval/out/systolic_matmul_4x4_w16_extracted.json; read_verilog eval/blackboxes/dsp_defs.v; synth_xilinx -family xcup; tee -o eval/out/systolic_matmul_4x4_w16_extracted.stat stat"

In [ ]:
!yosys -q -p "read_verilog eval/systolic/systolic_matmul_4x4_w32.v; proc; memory_map; opt; write_json eval/out/systolic_matmul_4x4_w32.json; synth_xilinx -family xcup; tee -o eval/out/systolic_matmul_4x4_w32.stat stat"

In [4]:
import emap
import json

SCHEMA_PATH = "emap/schema.sql"
TEST_NAME = "systolic_matmul_4x4_w32"

def simple_cost_model(type_: str, *ports) -> float:
    if type_ == "$dff":
        return len(ports[0]) * 1.0
    elif type_ in {"$muls", "$mulu"}:
        return len(ports[0]) * len(ports[1]) * 1.0
    elif type_ in {"$adds", "$addu", "$subs", "$subu"}:
        return min(len(ports[0]) + len(ports[1]), len(ports[2])) * 1.0
    return len(ports[0]) * 1.0  # other types

netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])
netlist.rebuild()

matches = emap.rewrites.ematch_wide_muls(netlist)
cnt = emap.rewrites.apply_wide_muls_split(netlist, matches)
print(f"Applied {cnt} rewrites")
netlist.rebuild()
matches = emap.rewrites.ematch_wide_dff(netlist)
cnt = emap.rewrites.apply_wide_dff_split(netlist, matches)
print(f"Applied {cnt} rewrites")
netlist.rebuild()

cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds", "$muls"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$muls"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$muls"])

    cnt = 0
    cnt += emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)

    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.techmap_dsp(netlist)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 64}, OutputFlag=False)
with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

Found 88 cells
Processing cell 0/88: $add$eval/systolic/systolic_matmul_4x4_w32.v:65$460
Database built with 5138 wires and global clock 2
Applied 16 rewrites
Applied 72 rewrites
Applied 327 rewrites
Applied 181 rewrites
Applied 50 rewrites
Applied 6 rewrites
No rewrites applied, stopping
C++ backend emapcc found
Removed 370 dominated cells, 1359 remain
C++ backend emapcc found
Grouped 17424 wires into 538 groups
ILP model solved with objective value: 8548.0


In [ ]:
!yosys -q -p "read_json eval/out/systolic_matmul_4x4_w32_extracted.json; read_verilog eval/blackboxes/dsp_defs.v; synth_xilinx -family xcup; tee -o eval/out/systolic_matmul_4x4_w32_extracted.stat stat"

In [ ]:
!yosys -q -p "read_verilog eval/systolic/systolic_matmul_8x8_w8.v; proc; memory_map; opt; write_json eval/out/systolic_matmul_8x8_w8.json; synth_xilinx -family xcup; tee -o eval/out/systolic_matmul_8x8_w8.stat stat"

In [ ]:
TEST_NAME = "systolic_matmul_8x8_w8"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()

cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds", "$muls"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$muls"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$muls"])

    cnt = 0
    cnt += emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)

    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

cnt = emap.rewrites.rewrite_sdff(netlist) # rewrite $dff to $sdff
print(f"Applied {cnt} rewrites")
# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.rewrite_tech(netlist, dsp_rules)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 64}, OutputFlag=False)
with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

In [ ]:
!yosys -q -p "read_json eval/out/systolic_matmul_8x8_w8_extracted.json; read_verilog eval/blackboxes/dsp_defs.v; synth_xilinx -family xcup; tee -o eval/out/systolic_matmul_8x8_w8_extracted.stat stat"

In [ ]:
!yosys -q -p "read_verilog eval/systolic/systolic_matmul_8x8_w16.v; proc; memory_map; opt; write_json eval/out/systolic_matmul_8x8_w16.json; synth_xilinx -family xcup; tee -o eval/out/systolic_matmul_8x8_w16.stat stat"

In [ ]:
TEST_NAME = "systolic_matmul_8x8_w16"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()

cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds", "$muls"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$muls"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$muls"])

    cnt = 0
    cnt += emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)

    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

cnt = emap.rewrites.rewrite_sdff(netlist) # rewrite $dff to $sdff
print(f"Applied {cnt} rewrites")
# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.rewrite_tech(netlist, dsp_rules)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 64}, OutputFlag=False)
with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

In [ ]:
!yosys -q -p "read_json eval/out/systolic_matmul_8x8_w16_extracted.json; read_verilog eval/blackboxes/dsp_defs.v; synth_xilinx -family xcup; tee -o eval/out/systolic_matmul_8x8_w16_extracted.stat stat"

In [ ]:
!yosys -q -p "read_verilog eval/systolic/systolic_matmul_8x8_w32.v; proc; memory_map; opt; write_json eval/out/systolic_matmul_8x8_w32.json; synth_xilinx -family xcup; tee -o eval/out/systolic_matmul_8x8_w32.stat stat"

In [ ]:
# TODO: better provide more dsp rules
TEST_NAME = "systolic_matmul_8x8_w32"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()
matches = emap.rewrites.ematch_wide_muls(netlist)
cnt = emap.rewrites.apply_wide_muls_split(netlist, matches)
print(f"Applied {cnt} rewrites")
netlist.rebuild()
matches = emap.rewrites.ematch_wide_dff(netlist)
cnt = emap.rewrites.apply_wide_dff_split(netlist, matches)
print(f"Applied {cnt} rewrites")
netlist.rebuild()

cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds", "$muls"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$muls"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$muls"])

    cnt = 0
    cnt += emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)

    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

cnt = emap.rewrites.rewrite_sdff(netlist) # rewrite $dff to $sdff
print(f"Applied {cnt} rewrites")
# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.rewrite_tech(netlist, dsp_rules)

# with open("debug.json", "w") as f:
#     json.dump(netlist.dump_tables(), f, indent=2)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 192}, OutputFlag=False)
with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

In [5]:
!yosys -q -p "read_json eval/out/systolic_matmul_8x8_w32_extracted.json; read_verilog eval/blackboxes/dsp_defs.v; synth_xilinx -family xcup; tee -o eval/out/systolic_matmul_8x8_w32_extracted.stat stat"

In [ ]:
!yosys -q -p "read_verilog eval/systolic/systolic_matmul_16x16_w8.v; proc; memory_map; opt; write_json eval/out/systolic_matmul_16x16_w8.json; synth_xilinx -family xcup; tee -o eval/out/systolic_matmul_16x16_w8.stat stat"

In [ ]:
TEST_NAME = "systolic_matmul_16x16_w8"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()

cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds", "$muls"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$muls"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$muls"])

    cnt = 0
    cnt += emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)

    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

cnt = emap.rewrites.rewrite_sdff(netlist) # rewrite $dff to $sdff
print(f"Applied {cnt} rewrites")
# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.rewrite_tech(netlist, dsp_rules)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 256}, OutputFlag=False)
with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

In [ ]:
!yosys -q -p "read_json eval/out/systolic_matmul_16x16_w8_extracted.json; read_verilog eval/blackboxes/dsp_defs.v; synth_xilinx -family xcup; tee -o eval/out/systolic_matmul_16x16_w8_extracted.stat stat"

In [ ]:
!yosys -q -p "read_verilog eval/systolic/systolic_matmul_16x16_w16.v; proc; memory; opt; write_json eval/out/systolic_matmul_16x16_w16.json; synth_xilinx -family xcup; tee -o eval/out/systolic_matmul_16x16_w16.stat stat"

In [ ]:
TEST_NAME = "systolic_matmul_16x16_w16"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()

cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds", "$muls"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$muls"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$muls"])

    cnt = 0
    cnt += emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)

    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

cnt = emap.rewrites.rewrite_sdff(netlist) # rewrite $dff to $sdff
print(f"Applied {cnt} rewrites")
# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.rewrite_tech(netlist, dsp_rules)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 256}, OutputFlag=False)
with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

In [ ]:
!yosys -q -p "read_json eval/out/systolic_matmul_16x16_w16_extracted.json; read_verilog eval/blackboxes/dsp_defs.v; synth_xilinx -family xcup; tee -o eval/out/systolic_matmul_16x16_w16_extracted.stat stat"

In [ ]:
!yosys -q -p "read_verilog eval/systolic/systolic_matmul_16x16_w32.v; proc; memory; opt; write_json eval/out/systolic_matmul_16x16_w32.json; synth_xilinx -family xcup; tee -o eval/out/systolic_matmul_16x16_w32.stat stat"

In [ ]:
import emap
import json

SCHEMA_PATH = "emap/schema.sql"
TEST_NAME = "systolic_matmul_16x16_w32"

def simple_cost_model(type_: str, *ports) -> float:
    if type_ == "$dff":
        return len(ports[0]) * 1.0
    elif type_ in {"$muls", "$mulu"}:
        return len(ports[0]) * len(ports[1]) * 1.0
    elif type_ in {"$adds", "$addu", "$subs", "$subu"}:
        return min(len(ports[0]) + len(ports[1]), len(ports[2])) * 1.0
    return len(ports[0]) * 1.0  # other types

netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])
netlist.rebuild()

matches = emap.rewrites.ematch_wide_muls(netlist)
cnt = emap.rewrites.apply_wide_muls_split(netlist, matches)
print(f"Applied {cnt} rewrites")
netlist.rebuild()
matches = emap.rewrites.ematch_wide_dff(netlist)
cnt = emap.rewrites.apply_wide_dff_split(netlist, matches)
print(f"Applied {cnt} rewrites")
netlist.rebuild()

cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds", "$muls"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$muls"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$muls"])

    cnt = 0
    cnt += emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)

    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.techmap_dsp(netlist)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 768}, OutputFlag=False)
with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

In [ ]:
!yosys -q -p "read_json eval/out/systolic_matmul_16x16_w32_extracted.json; read_verilog eval/blackboxes/dsp_defs.v; synth_xilinx -family xcup; tee -o eval/out/systolic_matmul_16x16_w32_extracted.stat stat"

### MLP

In [ ]:
import emap
import json

SCHEMA_PATH = "emap/schema.sql"

def simple_cost_model(type_: str, *ports) -> float:
    if type_ == "$dff":
        return len(ports[0]) * 1.0
    elif type_ in {"$muls", "$mulu"}:
        return len(ports[0]) * len(ports[1]) * 2.0
    elif type_ in {"$adds", "$addu", "$subs", "$subu"}:
        return min(len(ports[0]) + len(ports[1]), len(ports[2])) * 1.0
    return len(ports[0]) * 1.0  # other types

mlp_rules = {
    "pe_mac": {
        "requirements": {
            "pe_achronix": 1
        },
        "hidden_inputs": ["clk"],
        "inputs": ["a", "b", "rst"],
        "outputs": ["a_out", "b_out", "c"],
        "match_sql": """
            SELECT dff_a.d, dff_b.d, sdff_c.rst, dff_a.q, dff_b.q, add1.y
            FROM dffs AS dff_a JOIN dffs AS dff_b JOIN aby_cells AS mul1 JOIN sdffs AS sdff_c JOIN aby_cells AS add1
            ON dff_a.q = mul1.a AND dff_b.q = mul1.b AND mul1.y = add1.b AND sdff_c.q = add1.a AND sdff_c.d = add1.y
            WHERE mul1.type = '$muls' AND add1.type = '$adds'
                AND width_of(mul1.a) <= 16 AND width_of(mul1.b) <= 16 AND width_of(add1.b) <= 32
                AND width_of(add1.a) <= 32 AND width_of(add1.y) <= 32
        """
    },
    "2x2_mesh_mac": {
        "requirements": {
            "mlp_achronix": 1
        },
        "hidden_inputs": ["clk"],
        "inputs": [
            "a00", "a10",
            "b00", "b01",
            "rst00", "rst01", "rst10", "rst11"
        ],
        "outputs": [
            "a_out01", "a_out11",
            "b_out10", "b_out11",
            "c00", "c01", "c10", "c11"
        ],
        "match_sql": """
            SELECT pe00.a, pe10.a, pe00.b, pe01.b,
                pe00.rst, pe01.rst, pe10.rst, pe11.rst,
                pe01.a_out, pe11.a_out, pe10.b_out, pe11.b_out,
                pe00.c, pe01.c, pe10.c, pe11.c
            FROM tech_pe_mac AS pe00 JOIN tech_pe_mac AS pe01 JOIN tech_pe_mac AS pe10 JOIN tech_pe_mac AS pe11
            ON pe00.a_out = pe01.a AND pe00.b_out = pe10.b AND pe01.b_out = pe11.b AND pe10.a_out = pe11.a
        """
    },
    "4x4_mesh_mac": {
        "requirements": {
            "mlp_large_achronix": 1
        },
        "hidden_inputs": ["clk"],
        "inputs": [
            "a00", "a10", "a20", "a30",
            "b00", "b01", "b02", "b03",
            "rst00", "rst01", "rst02", "rst03",
            "rst10", "rst11", "rst12", "rst13",
            "rst20", "rst21", "rst22", "rst23",
            "rst30", "rst31", "rst32", "rst33"
        ],
        "outputs": [
            "a_out03", "a_out13", "a_out23", "a_out33",
            "b_out30", "b_out31", "b_out32", "b_out33",
            "c00", "c01", "c02", "c03",
            "c10", "c11", "c12", "c13",
            "c20", "c21", "c22", "c23",
            "c30", "c31", "c32", "c33"
        ],
        "match_sql": """
            SELECT mesh00.a00, mesh00.a10, mesh10.a00, mesh10.a10,
                mesh00.b00, mesh00.b01, mesh01.b00, mesh01.b01,
                mesh00.rst00, mesh00.rst01, mesh01.rst00, mesh01.rst01,
                mesh00.rst10, mesh00.rst11, mesh01.rst10, mesh01.rst11,
                mesh10.rst00, mesh10.rst01, mesh11.rst00, mesh11.rst01,
                mesh10.rst10, mesh10.rst11, mesh11.rst10, mesh11.rst11,
                mesh01.a_out01, mesh01.a_out11, mesh11.a_out01, mesh11.a_out11,
                mesh10.b_out10, mesh10.b_out11, mesh11.b_out10, mesh11.b_out11,
                mesh00.c00, mesh00.c01, mesh01.c00, mesh01.c01,
                mesh00.c10, mesh00.c11, mesh01.c10, mesh01.c11,
                mesh10.c00, mesh10.c01, mesh11.c00, mesh11.c01,
                mesh10.c10, mesh10.c11, mesh11.c10, mesh11.c11
            FROM tech_2x2_mesh_mac AS mesh00 JOIN tech_2x2_mesh_mac AS mesh01 JOIN tech_2x2_mesh_mac AS mesh10 JOIN tech_2x2_mesh_mac AS mesh11
            ON mesh00.a_out01 = mesh01.a00 AND mesh00.a_out11 = mesh01.a10
                AND mesh00.b_out10 = mesh10.b00 AND mesh00.b_out11 = mesh10.b01
                AND mesh01.b_out10 = mesh11.b00 AND mesh01.b_out11 = mesh11.b01
                AND mesh10.a_out01 = mesh11.a00 AND mesh10.a_out11 = mesh11.a10
        """
    }
}

In [ ]:
TEST_NAME = "systolic_matmul_16x16_w16"
SCHEMA_PATH = "emap/schema.sql"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()

cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$muls"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$muls"])

    cnt = 0
    cnt += emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)

    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

cnt = emap.rewrites.rewrite_sdff(netlist) # rewrite $dff to $sdff
print(f"Applied {cnt} rewrites")
# techmapping
emap.rewrites.create_tech_tables(netlist, mlp_rules)
emap.rewrites.rewrite_tech(netlist, mlp_rules)


mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, mlp_rules, {"pe_achronix": 0, "mlp_achronix": 0, "large_mlp_achronix": 16}, OutputFlag=False)
with open(f"eval/out/{TEST_NAME}_pe_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"systolic": mod}}, f, indent=2)

In [ ]:
TEST_NAME = "systolic_matmul_4x4_w8"
SCHEMA_PATH = "emap/schema.sql"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()

cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$muls"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$muls"])

    cnt = 0
    cnt += emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)

    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

cnt = emap.rewrites.rewrite_sdff(netlist) # rewrite $dff to $sdff
print(f"Applied {cnt} rewrites")
# techmapping
emap.rewrites.create_tech_tables(netlist, mlp_rules)
emap.rewrites.rewrite_tech(netlist, mlp_rules)


mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, mlp_rules, {"pe_achronix": 0, "mlp_achronix": 4}, OutputFlag=False)
with open(f"eval/out/{TEST_NAME}_2x2_mesh_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"systolic": mod}}, f, indent=2)

In [ ]:
TEST_NAME = "systolic_matmul_4x4_w8"
SCHEMA_PATH = "emap/schema.sql"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()

cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$muls"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$muls"])

    cnt = 0
    cnt += emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)

    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

cnt = emap.rewrites.rewrite_sdff(netlist) # rewrite $dff to $sdff
print(f"Applied {cnt} rewrites")
# techmapping
emap.rewrites.create_tech_tables(netlist, mlp_rules)
emap.rewrites.rewrite_tech(netlist, mlp_rules)


mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, mlp_rules, {"pe_achronix": 0, "mlp_achronix": 0, "mlp_large_achronix": 1}, OutputFlag=False)
with open(f"eval/out/{TEST_NAME}_4x4_mesh_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"systolic": mod}}, f, indent=2)